In [1]:
import os

PROJECT_DIR = (
    "/work/archcomplab/ssaraf/"
    "Maya Archarological Sites/Model Training/"
    "swin3d_project"
)

os.chdir(PROJECT_DIR)

print("Working directory:", os.getcwd())

Working directory: /work/archcomplab/ssaraf/Maya Archarological Sites/Model Training/swin3d_project


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from models.swin3d_model import Model
from datasets.belize_laz_dataset import create_dataloaders
from evaluate import evaluate_best_model

from visualization.vis import spatial_filter_predictions


/work/archcomplab/ssaraf/envs/maya_env/lib/python3.10/site-packages/MinkowskiEngine/__init__.py:36: UserWarning: The environment variable `OMP_NUM_THREADS` not set. MinkowskiEngine will automatically set `OMP_NUM_THREADS=16`. If you want to set `OMP_NUM_THREADS` manually, please export it on the command line before running a python script. e.g. `export OMP_NUM_THREADS=12; python your_program.py`. It is recommended to set it below 24.
  warnings.warn(


In [3]:
train_loader, validation_loader, test_loader = create_dataloaders(
    split_dir="dataset_splits",
    batch_size=8,
    voxel_max=65_000,
    num_workers=4,
)

print("Test batches:", len(test_loader))

Test batches: 10


In [4]:
train_loader, validation_loader, test_loader = create_dataloaders(
    split_dir="dataset_splits",
    batch_size=2,
    voxel_max=65_000,
    num_workers=4,
)

In [5]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = Model(
    num_classes=3,
    in_channels=4,
).to(device)

print("Device:", device)

Device: cuda


In [6]:
BEST_CHECKPOINT = Path(
    "/mnt/nrdstor/archcomplab/ssaraf/"
    "swin3d_checkpoints/"
    "dice_vicinity/"
    "swin3d_dice_vicinity_best_02.pth"
)

RESULTS_DIR = Path(
    "/mnt/nrdstor/archcomplab/ssaraf/"
    "swin3d_results/dice_vicinity"
)

print("Checkpoint exists:", BEST_CHECKPOINT.exists())
print("Checkpoint:", BEST_CHECKPOINT)
print("Results directory:", RESULTS_DIR)

Checkpoint exists: True
Checkpoint: /mnt/nrdstor/archcomplab/ssaraf/swin3d_checkpoints/dice_vicinity/swin3d_dice_vicinity_best_02.pth
Results directory: /mnt/nrdstor/archcomplab/ssaraf/swin3d_results/dice_vicinity


In [7]:
(
    metrics,
    results_df,
    per_class_df,
    confusion_matrix_df,
    saved_paths,
) = evaluate_best_model(
    model=model,
    test_loader=test_loader,
    checkpoint_path=BEST_CHECKPOINT,
    output_dir=RESULTS_DIR,
    device=device,
    class_names=[
        "Ground",
        "Structure",
        "Plazuela",
    ],
    print_progress=True,
    save_results=True,
)

Loaded best checkpoint from:
/mnt/nrdstor/archcomplab/ssaraf/swin3d_checkpoints/dice_vicinity/swin3d_dice_vicinity_best_02.pth
Checkpoint epoch: 55


/work/archcomplab/ssaraf/envs/maya_env/lib/python3.10/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/work/archcomplab/ssaraf/software/Swin3D/Swin3D/modules/swin3d_layers.py:239: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:78.)
  offset = torch.cuda.IntTensor(offset)


Evaluated test batch 1/40
Evaluated test batch 5/40
Evaluated test batch 10/40
Evaluated test batch 15/40
Evaluated test batch 20/40
Evaluated test batch 25/40
Evaluated test batch 30/40
Evaluated test batch 35/40
Evaluated test batch 40/40

BEST-CHECKPOINT TEST RESULTS
Best epoch:        55
Overall Accuracy:  0.935373
F1 Average:        0.550876
Precision Average: 0.543751
Recall Average:    0.582634
mIoU:              0.461278

Ground
  Precision: 0.980325
  Recall:    0.955762
  F1:        0.967888
  IoU:       0.937774
  Support:   4,897,010
  Confusion: TP=4,680,376, FP=93,935, FN=216,634, TN=209,055

Structure
  Precision: 0.208048
  Recall:    0.117161
  F1:        0.149905
  IoU:       0.081025
  Support:   37,555
  Confusion: TP=4,400, FP=16,749, FN=33,155, TN=5,145,696

Plazuela
  Precision: 0.442881
  Recall:    0.674979
  F1:        0.534835
  IoU:       0.365034
  Support:   265,435
  Confusion: TP=179,163, FP=225,377, FN=86,272, TN=4,709,188

Saved evaluation files:
  met

In [8]:
display(results_df)

,Test Set: Overall (Pixel / Voxel) Accuracy (OA),Test Set: F1 Average,Test Set: F1 - Ground,Test Set: F1 - Structure,Test Set: F1 - Plazuelas,Test Set: Precision Average,Test Set: Precision - Ground,Test Set: Precision - Structures,Test Set: Precision - Plazuelas,Test Set: Recall Average,Test Set: Recall - Ground,Test Set: Recall - Structure,Test Set: Recall - Plazuelas,Test Set: mIoU (mean Intersection over Union),Test Set: IoU Ground,Test Set: IoU Structure,Test Set: IoU Plazuela,Test Set: Confusion Matrix TP / FP / FN / TN: Ground,Test Set: Confusion Matrix TP / FP / FN / TN: Structures,Test Set: Confusion Matrix TP / FP / FN / TN: Plazuelas
0,0.935373,0.550876,0.967888,0.149905,0.534835,0.543751,0.980325,0.208048,0.442881,0.582634,0.955762,0.117161,0.674979,0.461278,0.937774,0.081025,0.365034,TP=4680376; FP=93935; FN=216634; TN=209055,TP=4400; FP=16749; FN=33155; TN=5145696,TP=179163; FP=225377; FN=86272; TN=4709188


In [9]:
display(per_class_df)

,Class,Precision,Recall,F1,IoU,TP,FP,FN,TN,Support
0,Ground,0.980325,0.955762,0.967888,0.937774,4680376,93935,216634,209055,4897010
1,Structure,0.208048,0.117161,0.149905,0.081025,4400,16749,33155,5145696,37555
2,Plazuela,0.442881,0.674979,0.534835,0.365034,179163,225377,86272,4709188,265435


In [10]:
display(confusion_matrix_df)

,Predicted Ground,Predicted Structure,Predicted Plazuela
Actual Ground,4680376,6678,209956
Actual Structure,17734,4400,15421
Actual Plazuela,76201,10071,179163


In [11]:
print(
    "Overall accuracy:",
    metrics["overall_accuracy"]
)

print(
    "Structure F1:",
    metrics["per_class"]["Structure"]["f1"]
)

print(
    "Plazuela IoU:",
    metrics["per_class"]["Plazuela"]["iou"]
)

Overall accuracy: 0.9353728846153846
Structure F1: 0.14990460615971654
Plazuela IoU: 0.36503386225275664


# Spatial Post-Processing: Before vs After

The same spatial filter used in the visualization notebook is applied to every test tile:

- 1 m XY grid
- 3×3 morphological opening
- Structure components smaller than 8 cells removed
- Plazuela components smaller than 15 cells removed
- Removed archaeological predictions are changed to Ground

The filter is applied **tile by tile** before recomputing the test metrics.


In [12]:
# ============================================================
# Load the best checkpoint explicitly
# ============================================================

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print(
    "Loaded checkpoint epoch:",
    checkpoint.get("epoch", "unknown"),
)


/tmp/ipykernel_66676/3042358863.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(


Loaded checkpoint epoch: 55


In [13]:
# ============================================================
# Filter settings
# ============================================================

FILTER_SETTINGS = {
    "cell_size": 1.0,
    "structure_min_cells": 8,
    "plazuela_min_cells": 15,
    "opening_iterations": 1,
    "closing_iterations": 0,
}

print(FILTER_SETTINGS)


{'cell_size': 1.0, 'structure_min_cells': 8, 'plazuela_min_cells': 15, 'opening_iterations': 1, 'closing_iterations': 0}


In [14]:
# ============================================================
# Helpers
# ============================================================

CLASS_NAMES = [
    "Ground",
    "Structure",
    "Plazuela",
]

NUM_CLASSES = 3


def add_to_confusion_matrix(
    confusion_matrix,
    labels,
    predictions,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    predictions = np.asarray(
        predictions,
        dtype=np.int64,
    )

    valid = (
        (labels >= 0)
        &
        (labels < NUM_CLASSES)
        &
        (predictions >= 0)
        &
        (predictions < NUM_CLASSES)
    )

    encoded = (
        NUM_CLASSES * labels[valid]
        +
        predictions[valid]
    )

    counts = np.bincount(
        encoded,
        minlength=NUM_CLASSES * NUM_CLASSES,
    )

    confusion_matrix += counts.reshape(
        NUM_CLASSES,
        NUM_CLASSES,
    )


def metrics_from_confusion_matrix(
    confusion_matrix,
):
    cm = confusion_matrix.astype(
        np.float64
    )

    total = cm.sum()

    overall_accuracy = (
        np.trace(cm) / total
        if total > 0
        else 0.0
    )

    rows = []

    for class_id, class_name in enumerate(
        CLASS_NAMES
    ):
        tp = cm[class_id, class_id]

        fn = (
            cm[class_id, :].sum()
            -
            tp
        )

        fp = (
            cm[:, class_id].sum()
            -
            tp
        )

        precision = (
            tp / (tp + fp)
            if (tp + fp) > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else 0.0
        )

        f1 = (
            2.0
            * precision
            * recall
            / (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )

        iou = (
            tp
            / (tp + fp + fn)
            if (tp + fp + fn) > 0
            else 0.0
        )

        rows.append(
            {
                "Class": class_name,
                "Precision": precision,
                "Recall": recall,
                "F1": f1,
                "IoU": iou,
                "GT Points": int(
                    cm[class_id, :].sum()
                ),
                "Predicted Points": int(
                    cm[:, class_id].sum()
                ),
                "True Positives": int(tp),
                "False Positives": int(fp),
                "False Negatives": int(fn),
            }
        )

    return overall_accuracy, pd.DataFrame(
        rows
    )


In [15]:
# ============================================================
# Evaluate all test tiles BEFORE and AFTER filtering
# ============================================================

confusion_before = np.zeros(
    (NUM_CLASSES, NUM_CLASSES),
    dtype=np.int64,
)

confusion_after = np.zeros(
    (NUM_CLASSES, NUM_CLASSES),
    dtype=np.int64,
)

total_removed_structure = 0
total_removed_plazuela = 0

tiles_processed = 0


with torch.no_grad():

    for batch_index, batch in enumerate(
        test_loader
    ):

        (
            coords,
            features,
            offset,
            labels,
            paths,
            metric_coords,
        ) = batch

        coords_gpu = coords.to(
            device,
            non_blocking=True,
        ).float()

        features_gpu = features.to(
            device,
            non_blocking=True,
        ).float()

        offset_gpu = offset.to(
            device,
            non_blocking=True,
        ).long()

        logits = model(
            (
                coords_gpu,
                features_gpu,
                offset_gpu,
            )
        )

        if isinstance(logits, tuple):
            logits = logits[0]

        predictions = (
            logits.argmax(dim=1)
            .cpu()
            .numpy()
        )

        coords_np = (
            coords.cpu()
            .numpy()
        )

        labels_np = (
            labels.cpu()
            .numpy()
        )

        # ----------------------------------------------------
        # Process each tile separately
        # ----------------------------------------------------

        start = 0

        for tile_in_batch, end in enumerate(
            offset.tolist()
        ):

            tile_coords = coords_np[
                start:end
            ]

            tile_labels = labels_np[
                start:end
            ]

            tile_predictions = predictions[
                start:end
            ]

            # BEFORE
            add_to_confusion_matrix(
                confusion_before,
                tile_labels,
                tile_predictions,
            )

            # AFTER spatial filtering
            (
                filtered_predictions,
                filter_info,
            ) = spatial_filter_predictions(
                coords=tile_coords,
                predictions=tile_predictions,
                **FILTER_SETTINGS,
                verbose=False,
            )

            add_to_confusion_matrix(
                confusion_after,
                tile_labels,
                filtered_predictions,
            )

            total_removed_structure += (
                filter_info[1][
                    "removed_points"
                ]
            )

            total_removed_plazuela += (
                filter_info[2][
                    "removed_points"
                ]
            )

            tiles_processed += 1

            start = end

        if (
            batch_index == 0
            or
            (batch_index + 1) % 10 == 0
            or
            batch_index + 1 == len(test_loader)
        ):
            print(
                f"Processed test batch "
                f"{batch_index + 1}/"
                f"{len(test_loader)}"
            )


print()
print(
    "Tiles processed:",
    tiles_processed,
)

print(
    "Structure predictions removed:",
    f"{total_removed_structure:,}",
)

print(
    "Plazuela predictions removed:",
    f"{total_removed_plazuela:,}",
)


Processed test batch 1/40
Processed test batch 10/40
Processed test batch 20/40
Processed test batch 30/40
Processed test batch 40/40

Tiles processed: 80
Structure predictions removed: 2,355
Plazuela predictions removed: 55,594


In [16]:
# ============================================================
# Calculate BEFORE and AFTER metrics
# ============================================================

(
    accuracy_before,
    per_class_before,
) = metrics_from_confusion_matrix(
    confusion_before
)

(
    accuracy_after,
    per_class_after,
) = metrics_from_confusion_matrix(
    confusion_after
)


print("=" * 65)
print("OVERALL ACCURACY")
print("=" * 65)

print(
    f"Before filter: {accuracy_before:.4f}"
)

print(
    f"After filter:  {accuracy_after:.4f}"
)

print(
    f"Change:        "
    f"{accuracy_after - accuracy_before:+.4f}"
)


OVERALL ACCURACY
Before filter: 0.9354
After filter:  0.9429
Change:        +0.0075


In [17]:
# ============================================================
# Per-class BEFORE / AFTER comparison
# ============================================================

before = per_class_before.copy()
after = per_class_after.copy()

comparison_df = pd.DataFrame({
    "Class": before["Class"],

    "Precision Before": before["Precision"],
    "Precision After": after["Precision"],

    "Recall Before": before["Recall"],
    "Recall After": after["Recall"],

    "F1 Before": before["F1"],
    "F1 After": after["F1"],

    "IoU Before": before["IoU"],
    "IoU After": after["IoU"],

    "FP Before": before["False Positives"],
    "FP After": after["False Positives"],

    "Predicted Before": before["Predicted Points"],
    "Predicted After": after["Predicted Points"],
})

display(
    comparison_df.style.format({
        "Precision Before": "{:.4f}",
        "Precision After": "{:.4f}",
        "Recall Before": "{:.4f}",
        "Recall After": "{:.4f}",
        "F1 Before": "{:.4f}",
        "F1 After": "{:.4f}",
        "IoU Before": "{:.4f}",
        "IoU After": "{:.4f}",
    })
)


,Class,Precision Before,Precision After,Recall Before,Recall After,F1 Before,F1 After,IoU Before,IoU After,FP Before,FP After,Predicted Before,Predicted After
0,Ground,0.9803,0.9783,0.9557,0.9654,0.9679,0.9718,0.9377,0.9451,94067,104812,4774256,4832205
1,Structure,0.2204,0.2380,0.1191,0.1136,0.1546,0.1538,0.0838,0.0833,15815,13665,20287,17932
2,Plazuela,0.4421,0.4901,0.6753,0.6460,0.5343,0.5574,0.3646,0.3864,226221,178389,405457,349863


In [18]:
# ============================================================
# Confusion matrices BEFORE / AFTER
# ============================================================

confusion_before_df = pd.DataFrame(
    confusion_before,
    index=[
        "GT Ground",
        "GT Structure",
        "GT Plazuela",
    ],
    columns=[
        "Pred Ground",
        "Pred Structure",
        "Pred Plazuela",
    ],
)

confusion_after_df = pd.DataFrame(
    confusion_after,
    index=[
        "GT Ground",
        "GT Structure",
        "GT Plazuela",
    ],
    columns=[
        "Pred Ground",
        "Pred Structure",
        "Pred Plazuela",
    ],
)

print("BEFORE FILTER")
display(confusion_before_df)

print()
print("AFTER FILTER")
display(confusion_after_df)


BEFORE FILTER


,Pred Ground,Pred Structure,Pred Plazuela
GT Ground,4680189,5864,210957
GT Structure,17819,4472,15264
GT Plazuela,76248,9951,179236



AFTER FILTER


,Pred Ground,Pred Structure,Pred Plazuela
GT Ground,4727393,4866,164751
GT Structure,19650,4267,13638
GT Plazuela,85162,8799,171474


In [19]:
# ============================================================
# Compact summary
# ============================================================

summary_df = pd.DataFrame([
    {
        "Version": "Before filter",
        "Overall Accuracy": accuracy_before,

        "Structure Precision": per_class_before.loc[
            per_class_before["Class"] == "Structure",
            "Precision",
        ].iloc[0],

        "Structure Recall": per_class_before.loc[
            per_class_before["Class"] == "Structure",
            "Recall",
        ].iloc[0],

        "Structure F1": per_class_before.loc[
            per_class_before["Class"] == "Structure",
            "F1",
        ].iloc[0],

        "Structure IoU": per_class_before.loc[
            per_class_before["Class"] == "Structure",
            "IoU",
        ].iloc[0],

        "Plazuela Precision": per_class_before.loc[
            per_class_before["Class"] == "Plazuela",
            "Precision",
        ].iloc[0],

        "Plazuela Recall": per_class_before.loc[
            per_class_before["Class"] == "Plazuela",
            "Recall",
        ].iloc[0],

        "Plazuela F1": per_class_before.loc[
            per_class_before["Class"] == "Plazuela",
            "F1",
        ].iloc[0],

        "Plazuela IoU": per_class_before.loc[
            per_class_before["Class"] == "Plazuela",
            "IoU",
        ].iloc[0],
    },

    {
        "Version": "After filter",
        "Overall Accuracy": accuracy_after,

        "Structure Precision": per_class_after.loc[
            per_class_after["Class"] == "Structure",
            "Precision",
        ].iloc[0],

        "Structure Recall": per_class_after.loc[
            per_class_after["Class"] == "Structure",
            "Recall",
        ].iloc[0],

        "Structure F1": per_class_after.loc[
            per_class_after["Class"] == "Structure",
            "F1",
        ].iloc[0],

        "Structure IoU": per_class_after.loc[
            per_class_after["Class"] == "Structure",
            "IoU",
        ].iloc[0],

        "Plazuela Precision": per_class_after.loc[
            per_class_after["Class"] == "Plazuela",
            "Precision",
        ].iloc[0],

        "Plazuela Recall": per_class_after.loc[
            per_class_after["Class"] == "Plazuela",
            "Recall",
        ].iloc[0],

        "Plazuela F1": per_class_after.loc[
            per_class_after["Class"] == "Plazuela",
            "F1",
        ].iloc[0],

        "Plazuela IoU": per_class_after.loc[
            per_class_after["Class"] == "Plazuela",
            "IoU",
        ].iloc[0],
    },
])

display(
    summary_df.style.format({
        column: "{:.4f}"
        for column in summary_df.columns
        if column != "Version"
    })
)


,Version,Overall Accuracy,Structure Precision,Structure Recall,Structure F1,Structure IoU,Plazuela Precision,Plazuela Recall,Plazuela F1,Plazuela IoU
0,Before filter,0.9354,0.2204,0.1191,0.1546,0.0838,0.4421,0.6753,0.5343,0.3646
1,After filter,0.9429,0.2380,0.1136,0.1538,0.0833,0.4901,0.6460,0.5574,0.3864
